# Spatial Autocorrelation with Geary's C

This notebook covers global Geary's C and its local counterpart as alternatives to Moran's I for detecting spatial autocorrelation.

## Learning goals

By the end of this notebook, you will be able to:

- explain how Geary's $C$ measures spatial autocorrelation differently from Moran's $I$
- interpret the range and direction of Geary's $C$
- compute global Geary's $C$ and assess its significance via permutation inference
- compute local Geary statistics and map the resulting cluster and outlier categories

In [ ]:
import geopandas as gpd
import libpysal as lps
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import esda

## Data preparation

We use the same Berlin neighbourhood data as the Moran's I notebooks, making it straightforward to compare results across statistics.

In [ ]:
gdf = gpd.read_file("data/berlin-neighbourhoods.geojson")

In [ ]:
bl_df = pd.read_csv("data/berlin-listings.csv")
geometry = gpd.points_from_xy(x=bl_df.longitude, y=bl_df.latitude, crs="epsg:4326")
bl_gdf = gpd.GeoDataFrame(bl_df, geometry=geometry)

In [ ]:
bl_gdf["price"] = bl_gdf["price"].astype("float32")
sj_gdf = gpd.sjoin(
    gdf, bl_gdf, how="inner", predicate="intersects", lsuffix="left", rsuffix="right"
)
median_price_gb = sj_gdf["price"].groupby([sj_gdf["neighbourhood_group"]]).mean()
gdf = gdf.join(median_price_gb, on="neighbourhood_group")
gdf.rename(columns={"price": "median_pri"}, inplace=True)
gdf["median_pri"] = gdf["median_pri"].fillna(gdf["median_pri"].mean())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10), subplot_kw={"aspect": "equal"})
gdf.plot(column="median_pri", scheme="Quantiles", k=5, cmap="GnBu", legend=True, ax=ax)
ax.set_axis_off()
ax.set_title("Median Airbnb price by neighbourhood (quintiles)")
plt.show()

## Why Geary's C?

Moran's $I$ measures spatial autocorrelation through the *cross-product* of mean-centered values: it is large when values above (or below) the mean cluster together. Geary's $C$ instead measures autocorrelation through the *squared difference* between neighbouring pairs:

$$C = \frac{(n-1)}{2S_0} \frac{\sum_i \sum_j w_{ij}(x_i - x_j)^2}{\sum_i (x_i - \bar{x})^2}$$

where $S_0$ is the sum of all weights.

**Interpreting $C$:**

| Value | Meaning |
|---|---|
| $C \approx 1$ | No spatial autocorrelation (null hypothesis) |
| $C < 1$ | Positive spatial autocorrelation (similar values cluster together) |
| $C > 1$ | Negative spatial autocorrelation (dissimilar values tend to be neighbours) |

Geary's $C$ is more sensitive to *local* variations between neighbours, whereas Moran's $I$ is a better overall summary of global spatial structure. The two statistics are inversely related: evidence of positive autocorrelation under Moran's $I$ ($I > E[I]$) corresponds to $C < 1$.

## Global Geary's C

We begin by building row-standardised Queen contiguity weights and computing the global statistic.

In [ ]:
df = gdf
wq = lps.weights.Queen.from_dataframe(df, use_index=False, silence_warnings=True)
wq.transform = "r"
y = df["median_pri"]

In [ ]:
np.random.seed(12345)
gc = esda.Geary(y, wq)
print(f"Geary's C:        {gc.C:.4f}")
print(f"Expected C:       {gc.EC:.4f}")
print(f"z-score (norm):   {gc.z_norm:.4f}")
print(f"p-value (norm):   {gc.p_norm:.4f}")
print(f"p-value (sim):    {gc.p_sim:.4f}")

A value of $C < 1$ together with a small p-value indicates statistically significant positive spatial autocorrelation — neighbourhoods with similar prices tend to be adjacent.

## Permutation-based inference

In [ ]:
import seaborn as sns

sns.kdeplot(gc.sim, fill=True)
plt.vlines(gc.C, 0, plt.gca().get_ylim()[1] * 0.9, color="r", label="Observed C")
plt.vlines(
    gc.EC,
    0,
    plt.gca().get_ylim()[1] * 0.9,
    color="k",
    linestyle="--",
    label="Expected C (=1)",
)
plt.xlabel("Geary's C")
plt.legend()
plt.title("Permutation reference distribution for global Geary's C")
plt.show()

## From global to local Geary

The local Geary statistic $c_i$ adapts the logic of global Geary to individual locations:

$$c_i = \sum_j w_{ij}(x_i - x_j)^2$$

- A **small** $c_i$ (relative to expectation) signals that location $i$ has *similar* neighbours — a local *cluster*.
- A **large** $c_i$ signals that location $i$ has *dissimilar* neighbours — a spatial *outlier*.

`Geary_Local` uses a scikit-learn-style estimator interface: first construct it, then call `.fit(y)`.

## Computing local Geary statistics

In [ ]:
np.random.seed(12345)
lg = esda.Geary_Local(connectivity=wq, labels=True, seed=12345).fit(y)
print(f"Significant locations (p<0.05): {(lg.p_sim < 0.05).sum()}")

The `labels` attribute uses the following coding when `labels=True`:

| Label code | Category |
|---|---|
| 1 | Outlier |
| 2 | Cluster |
| 3 | Other |
| 4 | Not significant |

A *cluster* location has a small $c_i$ — similar to its neighbours. An *outlier* location has a large $c_i$ — dissimilar to its neighbours.

In [ ]:
df = df.copy()
df["localC"] = lg.localG
df["p_sim"] = lg.p_sim
df["labels"] = lg.labs

label_map = {1: "Outlier", 2: "Cluster", 3: "Other", 4: "Not significant"}
color_map = {
    "Outlier": "red",
    "Cluster": "blue",
    "Other": "lightblue",
    "Not significant": "lightgrey",
}
df["label_str"] = df["labels"].map(label_map)
df["color"] = df["label_str"].map(color_map)

### Map of local $c_i$ values (continuous)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10), subplot_kw={"aspect": "equal"})
df.plot(column="localC", cmap="plasma_r", legend=True, ax=ax)
ax.set_axis_off()
ax.set_title("Local Geary $c_i$ (lower = more similar to neighbours)")
plt.show()

### Map of significant clusters and outliers

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 10), subplot_kw={"aspect": "equal"})

legend_handles = []

for label, color in color_map.items():
    subset = df[df["label_str"] == label]
    if not subset.empty:
        # Plot the actual geometry
        subset.plot(ax=ax, color=color)

        # Create a matching proxy artist for the legend
        patch = mpatches.Patch(color=color, label=label)
        legend_handles.append(patch)

# Configure the legend
# bbox_to_anchor(1.02, 1) moves it slightly right of the plot
ax.legend(
    handles=legend_handles,
    title="Cluster Types",
    loc="upper left",
    bbox_to_anchor=(1.02, 1),
    frameon=False,
)

ax.set_axis_off()
ax.set_title("Local Geary: clusters and outliers (p < 0.05)", fontsize=14, pad=20)

plt.tight_layout()
plt.show()

## Comparing Geary's C and Moran's I

Both statistics detect the same broad spatial structure, but they are not identical:

- **Moran's I** compares each observation to the *global mean*; it is more sensitive to broad trends.
- **Geary's C** compares each observation directly to its *neighbours*; it is more sensitive to local differences.

When the two disagree, it usually indicates that a local cluster or outlier is driven by a very local contrast rather than a deviation from the global level.

For most applied work Moran's I is the first choice; Geary's C and its local form add value when you specifically want to emphasize *pair-wise dissimilarity* between neighbours rather than global clustering.

## Takeaways

- Global Geary's $C$ is centred at 1 under the null; values below 1 indicate positive autocorrelation.
- Geary's $C$ is based on squared differences between neighbours, making it sensitive to local contrasts.
- Local Geary $c_i$ classifies each location as a cluster (similar neighbours) or outlier (dissimilar neighbours).
- Permutation inference is preferred over the normality assumption for small samples or non-normal data.
- Geary and Moran are complementary: using both gives a fuller picture of spatial structure.